In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SOURCE_PATH = "/Volumes/property_catalog/property_management/landing_zone/tenants.csv"
TARGET_SCHEMA = "property_catalog.property_management"

CLEAN_TABLE = f"{TARGET_SCHEMA}.clean_tenants"
REJECTED_TABLE = f"{TARGET_SCHEMA}.rejected_tenants"

In [0]:
def extract_tenants(source_path):
    tenants_df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .csv(source_path)
    )

    print("Tenants extracted successfully!")

    # Log the number of source records
    print(f"Tenants extracted: {tenants_df.count()} rows")

    return tenants_df

tenants_df = extract_tenants(SOURCE_PATH)

display(tenants_df)

In [0]:
def transform_tenants(tenants_df):
    # Standardise tenant columns and data types
    transformed_df = (
        tenants_df
        # Preserve the original lease end date for validation
        .withColumn("_raw_LeaseEndDate", F.col("LeaseEndDate"))
        .withColumn("Id", F.expr("try_cast(trim(Id) AS BIGINT)"))
        .withColumn("PropertyId", F.expr("try_cast(trim(PropertyId) AS BIGINT)"))
        .withColumn("FirstName", F.trim(F.col("FirstName")))
        .withColumn("LastName", F.trim(F.col("LastName")))
        .withColumn("Email", F.lower(F.trim(F.col("Email"))))
        .withColumn(
            "Phone",
            F.when(F.trim(F.col("Phone")) == "", F.lit(None).cast("string")).otherwise(F.trim(F.col("Phone")))
        )
        .withColumn("LeaseStartDate", F.expr("try_cast(trim(LeaseStartDate) AS DATE)"))
        .withColumn("LeaseEndDate", F.expr("try_cast(trim(LeaseEndDate) AS DATE)"))
    )

    print("Tenants transformation completed!")
    print(f"Total records: {transformed_df.count()}")

    display(transformed_df)

    return transformed_df

transformed_tenants_df = transform_tenants(tenants_df)

In [0]:

def validate_tenants(transformed_tenants_df):
    validated_df = (
        transformed_tenants_df
        # Flag duplicate tenant IDs
        .withColumn("duplicate_tenant_id", F.count("*").over(Window.partitionBy("Id")) > 1)

        # Flag records with missing required values
        .withColumn(
            "has_missing_values",
            F.col("Id").isNull()
            | F.col("FirstName").isNull()
            | (F.trim(F.col("FirstName")) == "")
            | F.col("LastName").isNull()
            | (F.trim(F.col("LastName")) == "")
            | F.col("Email").isNull()
            | (F.trim(F.col("Email")) == "")
            | F.col("LeaseStartDate").isNull()
            | F.col("PropertyId").isNull()
        )

        # Flag missing, empty or over-100-character first names
        .withColumn(
            "invalid_first_name",
            F.col("FirstName").isNull()
            | (F.trim(F.col("FirstName")) == "")
            | (F.length(F.col("FirstName")) > 100)
        )

        # Flag missing, empty or over-100-character last names
        .withColumn(
            "invalid_last_name",
            F.col("LastName").isNull()
            | (F.trim(F.col("LastName")) == "")
            | (F.length(F.col("LastName")) > 100)
        )

        # Flag missing or invalid email addresses
        .withColumn(
            "invalid_email",
            F.col("Email").isNull()
            | (~F.col("Email").rlike(
                r"^[A-Za-z0-9.!#$%&'*+/=?^_`{|}~-]+@[A-Za-z0-9-]+(\.[A-Za-z0-9-]+)+$"
            ))
        )

        # Flag missing or invalid lease start dates
        .withColumn("invalid_lease_start_date", F.col("LeaseStartDate").isNull())

        # Flag supplied but invalid lease end dates
        .withColumn("invalid_lease_end_date", F.col("_raw_LeaseEndDate").isNotNull() & F.col("LeaseEndDate").isNull())

        # Flag lease end dates before lease start dates
        .withColumn(
            "invalid_lease_period",
            F.col("LeaseStartDate").isNotNull() & F.col("LeaseEndDate").isNotNull() & (F.col("LeaseEndDate") < F.col("LeaseStartDate"))
        )

        # Flag missing or invalid property IDs
        .withColumn("invalid_property_id", F.col("PropertyId").isNull() | (F.col("PropertyId") <= 0))
    )

    print("Tenant validation completed!")

    return validated_df

validated_tenants_df = validate_tenants(transformed_tenants_df)

display(validated_tenants_df)

In [0]:
def separate_tenants(validated_tenants_df):
    rejection_condition = (
        F.col("duplicate_tenant_id")
        | F.col("has_missing_values")
        | F.col("invalid_first_name")
        | F.col("invalid_last_name")
        | F.col("invalid_email")
        | F.col("invalid_lease_start_date")
        | F.col("invalid_lease_end_date")
        | F.col("invalid_lease_period")
        | F.col("invalid_property_id")
    )

    # Filter rejected tenant records and add detailed reasons for each validation failure
    rejected_tenants_df = (
        validated_tenants_df
        .filter(rejection_condition)
        .withColumn(
            "rejection_reason",
            F.concat_ws(
                ". ",
                F.when(F.col("duplicate_tenant_id"), F.lit("Duplicate tenant ID")),
                F.when(F.col("has_missing_values"), F.lit("Missing required values")),
                F.when(F.col("invalid_first_name"), F.lit("First name is missing or exceeds 100 characters")),
                F.when(F.col("invalid_last_name"), F.lit("Last name is missing or exceeds 100 characters")),
                F.when(F.col("invalid_email"), F.lit("Invalid email address")),
                F.when(F.col("invalid_lease_start_date"), F.lit("Invalid lease start date")),
                F.when(F.col("invalid_lease_end_date"), F.lit("Invalid lease end date")),
                F.when(F.col("invalid_lease_period"), F.lit("Lease end date is before lease start date")),
                F.when(F.col("invalid_property_id"), F.lit("Property ID must be a positive integer"))
            )
        )
    )

    # Filter clean tenant records
    clean_tenants_df = validated_tenants_df.filter(~rejection_condition)

    return clean_tenants_df, rejected_tenants_df

clean_tenants_df, rejected_tenants_df = separate_tenants(validated_tenants_df)

print("Clean tenants:")
display(clean_tenants_df)

print("Rejected tenants:")
display(rejected_tenants_df)


In [0]:
def summarise_tenants(tenants_df, clean_tenants_df, rejected_tenants_df):
    total_count = tenants_df.count()
    clean_count = clean_tenants_df.count()
    rejected_count = rejected_tenants_df.count()

    # Calculate the clean record success rate
    success_rate = (
        clean_count / total_count * 100
        if total_count > 0
        else 0
    )

    print("=" * 45)
    print("TENANTS ETL SUMMARY")
    print("=" * 45)
    print(f"Source: {SOURCE_PATH}")
    print(f"Total records: {total_count}")
    print(f"Clean records: {clean_count}")
    print(f"Rejected records: {rejected_count}")
    print(f"Success rate: {success_rate:.2f}%")
    print("=" * 45)

    return {
        "total": total_count,
        "clean": clean_count,
        "rejected": rejected_count,
        "success_rate": success_rate
    }

tenants_summary = summarise_tenants(tenants_df, clean_tenants_df, rejected_tenants_df)

In [0]:
def load_tenants(clean_tenants_df, rejected_tenants_df):
    # Remove validation flags from the clean output
    clean_tenants_output_df = clean_tenants_df.drop(
        "duplicate_tenant_id",
        "has_missing_values",
        "invalid_first_name",
        "invalid_last_name",
        "invalid_email",
        "invalid_lease_start_date",
        "invalid_lease_end_date",
        "invalid_lease_period",
        "invalid_property_id"
    )
    
    # Save clean tenants to a managed Delta table
    (
        clean_tenants_output_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(CLEAN_TABLE)
    )

    # Save rejected tenants to a managed Delta table
    (
        rejected_tenants_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(REJECTED_TABLE)
    )

    print(f"Clean tenants records saved to: {CLEAN_TABLE}")
    print(f"Rejected tenants records saved to: {REJECTED_TABLE}")

load_tenants(clean_tenants_df,rejected_tenants_df)